## Imports

In [2]:
import glob
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

import time

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [4]:
## Connection
connection = presto.connect(
        host='processing-2.processing.data.production.internal' ,
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [5]:
local_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/'
local_path

'/Users/E2074/local-datasets/customer/segmentation/consideration/'

In [6]:
city = 'delhi'
city_name = 'Delhi'

## Datasets & Parameter

In [7]:
start_date = datetime.strptime("2025-03-03", "%Y-%m-%d")

run_date = [(start_date + timedelta(weeks=i)).strftime("%Y-%m-%d") for i in range(10)]
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [8]:
def get_feature_engineering(df):
    
    print('feature engineering part')
    # Adding required active month fields
    df['isAct_m01'] = np.where(df[['isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m02'] = np.where(df[['isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m03'] = np.where(df[['isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13']].sum(axis=1) >= 1, 1, 0)
    
    df['max_active_months'] = 3
    df['total_active_months'] = df[['isAct_m01', 'isAct_m02', 'isAct_m03']].sum(axis=1)
    
    df['days_since_1st_ride_in_3months'] = np.where(df['rapido_firstride_age']>=91,91,df['rapido_firstride_age'])
    df['weeks_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/7)
    df['months_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/31)

    df["customerid"] = df["customerid"].fillna('NA')
    df.fillna(0,inplace=True)


    # Adding required derived percentage fields for making thresholds
    df['active_days_percet'] = (df['total_active_days']/df['days_since_1st_ride_in_3months']).round(2)
    df['active_weeks_percet'] = (df['total_active_weeks']/df['weeks_since_1st_ride_in_3months']).round(2)
    df['active_months_percet'] = (df['total_active_months']/df['months_since_1st_ride_in_3months']).round(2)

    return df

In [9]:
def get_segment(df):

    print('segmentation part')
    
    df['hh_monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['active_months_percet'] >= 0.66), '1-HH-MONTHLY', None)
    df['hh_quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['total_active_days'] > 0), '2-HH-QUARTERLY', None)
    df['hh_inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])), '3-HH-INACTIVE',  None)
    
    df['daily_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH']))& (df['active_days_percet'] >= 0.45), '4-DAILY', None)
    df['weekly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_weeks_percet'] >= 0.70), '5-WEEKLY', None)
    df['monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_months_percet'] >= 0.66), '6-MONTHLY', None)
    df['quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & df['total_active_days'] > 0, '7-QUARTERLY', None)
    df['inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['total_active_days'] == 0), '8-INACTIVE', None)
    df['new_segment'] = np.where((~df['taxi_ltr_segment'].isin(['HH', 'PHH'])), '9-NEW', None)
    df['segment'] = df['hh_monthly_segment']\
                        .fillna(df['hh_quarterly_segment'])\
                        .fillna(df['hh_inactive_segment'])\
                        .fillna(df['daily_segment'])\
                        .fillna(df['weekly_segment'])\
                        .fillna(df['monthly_segment'])\
                        .fillna(df['quarterly_segment'])\
                        .fillna(df['inactive_segment'])\
                        .fillna(df['new_segment'])\
                        .fillna('check')
    
    df['updated_segment'] = df['segment']
    
    df.loc[((df['updated_segment']=='7-QUARTERLY')& (df['weeks_since_1st_ride_in_3months']<=8)), 'updated_segment'] = '6-MONTHLY'
    df.loc[((df['updated_segment'] == '5-WEEKLY') & (df['weeks_since_1st_ride_in_3months']<=2)), 'updated_segment'] = '6-MONTHLY'

    return df

In [10]:
df_list

NameError: name 'df_list' is not defined

In [11]:
for df_name in df_list:
    print(df_name)
    df = dfs[df_name]
    df_temp = get_feature_engineering(df)
    df_final = get_segment(df_temp)
    df_final.to_parquet(local_path + city + '/segment/{}.parquet'.format(df_name), index=False)
    
    print('---')

NameError: name 'df_list' is not defined

In [12]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/' + city + '/segment/test/'

# Get all parquet files in the folder
parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

In [13]:
parquet_files = sorted(parquet_files)
parquet_files

['/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250303.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250310.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250317.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250324.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250331.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250407.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250414.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250421.parquet',
 '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250428.parquet',
 '/Users/E2074/local-datasets/custome

In [14]:
# Read and concatenate all parquet files into a single DataFrame
df_segment_concat = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

In [15]:
df_segment_concat.head(1)

,run_date,customer_id,segment,updated_segment
0,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE


In [16]:
df_segment_concat.shape

(95649138, 4)

In [17]:
df_segment_concat.segment.unique()

array(['3-HH-INACTIVE', '2-HH-QUARTERLY', '1-HH-MONTHLY', '6-MONTHLY',
       '7-QUARTERLY', '5-WEEKLY', '4-DAILY'], dtype=object)

## Stability

In [18]:
df_segment_concat.run_date.unique()

array(['2025-03-03', '2025-03-10', '2025-03-17', '2025-03-24',
       '2025-03-31', '2025-04-07', '2025-04-14', '2025-04-21',
       '2025-04-28', '2025-05-05'], dtype=object)

In [19]:
df_base = df_segment_concat[df_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base

,run_date,customer_id,segment,updated_segment
0,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE
1,2025-03-03,65953febb6c2d0503fbb8c1c,2-HH-QUARTERLY,2-HH-QUARTERLY
2,2025-03-03,670da09b0401f7fbd582e8ba,3-HH-INACTIVE,3-HH-INACTIVE
3,2025-03-03,67205781aa7eb8928ebbe454,3-HH-INACTIVE,3-HH-INACTIVE
4,2025-03-03,62fb9bc8c496e55cdb09a787,3-HH-INACTIVE,3-HH-INACTIVE
...,...,...,...,...
9106563,2025-03-03,65b05f9c706706ae5505ac91,7-QUARTERLY,7-QUARTERLY
9106564,2025-03-03,646477b619489e42cdcbdd69,6-MONTHLY,6-MONTHLY
9106565,2025-03-03,5d8a4485941a7d1c51c1b779,5-WEEKLY,5-WEEKLY
9106566,2025-03-03,5c39ec214a267149c76dee56,5-WEEKLY,5-WEEKLY


In [20]:
df_stability_window = df_segment_concat[~df_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window

,run_date,customer_id,segment,updated_segment
0,2025-03-10,672b3dd7c465a3565e785770,1-HH-MONTHLY,1-HH-MONTHLY
1,2025-03-10,654ce60924b813830ee4b720,6-MONTHLY,6-MONTHLY
2,2025-03-10,66897198bdbf8155016aa634,6-MONTHLY,6-MONTHLY
3,2025-03-10,656052879ef2f505d955cf5f,5-WEEKLY,5-WEEKLY
4,2025-03-10,624417cdf42d275568a7f69e,7-QUARTERLY,7-QUARTERLY
...,...,...,...,...
86542565,2025-05-05,67594e59feadb698c3c75fa2,3-HH-INACTIVE,3-HH-INACTIVE
86542566,2025-05-05,64dad77aeb25685129bd04d6,3-HH-INACTIVE,3-HH-INACTIVE
86542567,2025-05-05,654a01704ced0fce02f621d3,3-HH-INACTIVE,3-HH-INACTIVE
86542568,2025-05-05,62e2ced6d8a2495544aec6bf,3-HH-INACTIVE,3-HH-INACTIVE


In [21]:
df_stability_window.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [22]:
df_merge = pd.merge(df_base, df_stability_window, how='left', on='customer_id')
df_merge

,run_date_x,customer_id,segment_x,updated_segment_x,run_date_y,segment_y,updated_segment_y
0,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE,2025-03-10,3-HH-INACTIVE,3-HH-INACTIVE
1,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE,2025-03-17,3-HH-INACTIVE,3-HH-INACTIVE
2,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE,2025-03-24,3-HH-INACTIVE,3-HH-INACTIVE
3,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE,2025-03-31,3-HH-INACTIVE,3-HH-INACTIVE
4,2025-03-03,63145a0640c18148020ff5e5,3-HH-INACTIVE,3-HH-INACTIVE,2025-04-07,3-HH-INACTIVE,3-HH-INACTIVE
...,...,...,...,...,...,...,...
77768511,2025-03-03,6651bb27ff7759169f1cc345,7-QUARTERLY,7-QUARTERLY,2025-04-07,7-QUARTERLY,7-QUARTERLY
77768512,2025-03-03,6651bb27ff7759169f1cc345,7-QUARTERLY,7-QUARTERLY,2025-04-14,7-QUARTERLY,7-QUARTERLY
77768513,2025-03-03,6651bb27ff7759169f1cc345,7-QUARTERLY,7-QUARTERLY,2025-04-21,7-QUARTERLY,7-QUARTERLY
77768514,2025-03-03,6651bb27ff7759169f1cc345,7-QUARTERLY,7-QUARTERLY,2025-04-28,6-MONTHLY,6-MONTHLY


In [23]:
df_merge_pivot = df_merge.groupby(['run_date_x', 'run_date_y', 'updated_segment_x', 'updated_segment_y']).agg(customers=('customer_id','nunique')).reset_index()

In [24]:
df_merge_pivot

,run_date_x,run_date_y,updated_segment_x,updated_segment_y,customers
0,2025-03-03,2025-03-10,1-HH-MONTHLY,1-HH-MONTHLY,609229
1,2025-03-03,2025-03-10,1-HH-MONTHLY,2-HH-QUARTERLY,136995
2,2025-03-03,2025-03-10,1-HH-MONTHLY,3-HH-INACTIVE,42
3,2025-03-03,2025-03-10,1-HH-MONTHLY,4-DAILY,2838
4,2025-03-03,2025-03-10,1-HH-MONTHLY,5-WEEKLY,6683
...,...,...,...,...,...
319,2025-03-03,2025-05-05,6-MONTHLY,7-QUARTERLY,347930
320,2025-03-03,2025-05-05,7-QUARTERLY,4-DAILY,5143
321,2025-03-03,2025-05-05,7-QUARTERLY,5-WEEKLY,14298
322,2025-03-03,2025-05-05,7-QUARTERLY,6-MONTHLY,352927


In [25]:
df_merge_pivot.pivot(index = ['run_date_x', 'updated_segment_x', 'updated_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers             \
run_date_y                                     2025-03-10 2025-03-17   
run_date_x updated_segment_x updated_segment_y                         
2025-03-03 1-HH-MONTHLY      1-HH-MONTHLY        609229.0   483042.0   
                             2-HH-QUARTERLY      136995.0   236439.0   
                             3-HH-INACTIVE           42.0       72.0   
                             4-DAILY               2838.0     2095.0   
                             5-WEEKLY              6683.0    14209.0   
                             6-MONTHLY            22232.0    38429.0   
                             7-QUARTERLY            845.0     2272.0   
           2-HH-QUARTERLY    1-HH-MONTHLY         63046.0    91851.0   
                             2-HH-QUARTERLY      806700.0   672207.0   
                             3-HH-INACTIVE        87638.0   175284.0   
                             4-DAILY                 27.0       30.0   
                             5-WEEKLY               110.0      353.0   
                             6-MONTHLY             7270.0    16548.0   
                             7-QUARTERLY           9524.0    14895.0   
           3-HH-INACTIVE     1-HH-MONTHLY         13518.0    13072.0   
                             2-HH-QUARTERLY       38499.0    76096.0   
                             3-HH-INACTIVE      3582282.0  3529495.0   
                             4-DAILY                765.0      554.0   
                             5-WEEKLY                 0.0        0.0   
                             6-MONTHLY              435.0     1204.0   
                             7-QUARTERLY           7357.0    16114.0   
           4-DAILY           4-DAILY             167227.0   155965.0   
                             5-WEEKLY              9940.0    17809.0   
                             6-MONTHLY             3370.0     5489.0   
                             7-QUARTERLY              0.0        5.0   
           5-WEEKLY          4-DAILY               9741.0    13141.0   
                             5-WEEKLY            316760.0   283768.0   
                             6-MONTHLY            35205.0    61865.0   
                             7-QUARTERLY              0.0        5.0   
           6-MONTHLY         4-DAILY               2025.0     3613.0   
                             5-WEEKLY             33947.0    53230.0   
                             6-MONTHLY          1845484.0  1710946.0   
                             7-QUARTERLY         151240.0   247190.0   
           7-QUARTERLY       4-DAILY                  2.0       44.0   
                             5-WEEKLY                 0.0        0.0   
                             6-MONTHLY           138923.0   226599.0   
                             7-QUARTERLY         859683.0   688486.0   

                                                                      \
run_date_y                                     2025-03-24 2025-03-31   
run_date_x updated_segment_x updated_segment_y                         
2025-03-03 1-HH-MONTHLY      1-HH-MONTHLY        382462.0   307695.0   
                             2-HH-QUARTERLY      311261.0   363424.0   
                             3-HH-INACTIVE           91.0      114.0   
                             4-DAILY               1877.0     1766.0   
                             5-WEEKLY             18387.0    17696.0   
                             6-MONTHLY            57467.0    79430.0   
                             7-QUARTERLY           3198.0     2955.0   
           2-HH-QUARTERLY    1-HH-MONTHLY        100612.0    89358.0   
                             2-HH-QUARTERLY      579837.0   507405.0   
                             3-HH-INACTIVE       240723.0   307064.0   
                             4-DAILY                 26.0       18.0   
                             5-WEEKLY               274.0      207.0   
                             6-MONTHLY            28068.0    39863.0   
               

In [26]:
df_base.groupby(['run_date', 'updated_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,updated_segment,customers
0,2025-03-03,1-HH-MONTHLY,782084
1,2025-03-03,2-HH-QUARTERLY,978043
2,2025-03-03,3-HH-INACTIVE,3651762
3,2025-03-03,4-DAILY,182416
4,2025-03-03,5-WEEKLY,365821
5,2025-03-03,6-MONTHLY,2056708
6,2025-03-03,7-QUARTERLY,1089734


In [27]:
df_merge_pivot1 = df_merge.groupby(['run_date_x', 'run_date_y', 'segment_x', 'segment_y']).agg(customers=('customer_id','nunique')).reset_index()

In [28]:
df_merge_pivot1

,run_date_x,run_date_y,segment_x,segment_y,customers
0,2025-03-03,2025-03-10,1-HH-MONTHLY,1-HH-MONTHLY,609229
1,2025-03-03,2025-03-10,1-HH-MONTHLY,2-HH-QUARTERLY,136995
2,2025-03-03,2025-03-10,1-HH-MONTHLY,3-HH-INACTIVE,42
3,2025-03-03,2025-03-10,1-HH-MONTHLY,4-DAILY,2838
4,2025-03-03,2025-03-10,1-HH-MONTHLY,5-WEEKLY,9012
...,...,...,...,...,...
326,2025-03-03,2025-05-05,6-MONTHLY,7-QUARTERLY,344550
327,2025-03-03,2025-05-05,7-QUARTERLY,4-DAILY,5198
328,2025-03-03,2025-05-05,7-QUARTERLY,5-WEEKLY,14521
329,2025-03-03,2025-05-05,7-QUARTERLY,6-MONTHLY,356331


In [29]:
df_merge_pivot1.pivot(index = ['run_date_x', 'segment_x', 'segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers                        \
run_date_y                               2025-03-10 2025-03-17 2025-03-24   
run_date_x segment_x      segment_y                                         
2025-03-03 1-HH-MONTHLY   1-HH-MONTHLY     609229.0   483042.0   382462.0   
                          2-HH-QUARTERLY   136995.0   236439.0   311261.0   
                          3-HH-INACTIVE        42.0       72.0       91.0   
                          4-DAILY            2838.0     2095.0     1877.0   
                          5-WEEKLY           9012.0    14430.0    18387.0   
                          6-MONTHLY         19107.0    36846.0    55250.0   
                          7-QUARTERLY        1641.0     3634.0     5415.0   
           2-HH-QUARTERLY 1-HH-MONTHLY      63046.0    91851.0   100612.0   
                          2-HH-QUARTERLY   806700.0   672207.0   579837.0   
                          3-HH-INACTIVE     87638.0   175284.0   240723.0   
                          4-DAILY              27.0       30.0       26.0   
                          5-WEEKLY            110.0      359.0      274.0   
                          6-MONTHLY          7104.0    16402.0    27988.0   
                          7-QUARTERLY        9690.0    15035.0    18870.0   
           3-HH-INACTIVE  1-HH-MONTHLY      13518.0    13072.0    12500.0   
                          2-HH-QUARTERLY    38499.0    76096.0   107570.0   
                          3-HH-INACTIVE   3582282.0  3529495.0  3480697.0   
                          4-DAILY             765.0      554.0      454.0   
                          5-WEEKLY            298.0      635.0      716.0   
                          6-MONTHLY            96.0      482.0      949.0   
                          7-QUARTERLY        7398.0    16201.0    26964.0   
           4-DAILY        4-DAILY          167227.0   155965.0   150654.0   
                          5-WEEKLY          10543.0    17819.0    21150.0   
                          6-MONTHLY          2730.0     5230.0     7557.0   
                          7-QUARTERLY          37.0      254.0      764.0   
           5-WEEKLY       4-DAILY           10056.0    13368.0    19662.0   
                          5-WEEKLY         318949.0   286230.0   263424.0   
                          6-MONTHLY         35422.0    62617.0    79546.0   
                          7-QUARTERLY        1417.0      661.0     1199.0   
           6-MONTHLY      4-DAILY            1706.0     3376.0     6044.0   
                          5-WEEKLY          32025.0    50358.0    70613.0   
                          6-MONTHLY       1830794.0  1695858.0  1626689.0   
                          7-QUARTERLY      153333.0   250759.0   290865.0   
           7-QUARTERLY    4-DAILY               6.0       54.0      249.0   
                          5-WEEKLY             29.0      410.0      270.0   
                          6-MONTHLY        142808.0   231672.0   279395.0   
                          7-QUARTERLY      866465.0   693524.0   581104.0   

                                                                           \
run_date_y                               2025-03-31 2025-04-07 2025-04-14   
run_date_x segment_x      segment_y                                         
2025-03-03 1-HH-MONTHLY   1-HH-MONTHLY     307695.0   253187.0   224314.0   
                          2-HH-QUARTERLY   363424.0   380547.0   376702.0   
                          3-HH-INACTIVE       114.0    17289.0    29329.0   
                          4-DAILY            1766.0     1654.0     1658.0   
                          5-WEEKLY          17696.0    16740.0    17282.0   
                          6-MONTHLY         76438.0    98713.0   112484.0   
                          7-QUARTERLY        5947.0     3352.0     7911.0   
           2-HH-QUARTERLY 1-HH-MONTHLY      89358.0   102574.0    99732.0   
                          2-HH-QUARTERLY   507405.0   430810.0   370417.0   
                          3-HH-INACTI

In [30]:
df_base.groupby(['run_date', 'segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,segment,customers
0,2025-03-03,1-HH-MONTHLY,782084
1,2025-03-03,2-HH-QUARTERLY,978043
2,2025-03-03,3-HH-INACTIVE,3651762
3,2025-03-03,4-DAILY,182416
4,2025-03-03,5-WEEKLY,370043
5,2025-03-03,6-MONTHLY,2041465
6,2025-03-03,7-QUARTERLY,1100755


## Previous Segment Stability

### taxi_regularity_segment

In [8]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [17]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        taxi_regularity_segment
        -- taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = '{city_name}'
        and taxi_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn3)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + city + '/prev_segment/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [18]:
for i in run_date[3:]:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-24
executed -  2025-03-24
starting -  2025-03-31
executed -  2025-03-31
starting -  2025-04-07
executed -  2025-04-07
starting -  2025-04-14
executed -  2025-04-14
starting -  2025-04-21
executed -  2025-04-21
starting -  2025-04-28
executed -  2025-04-28
starting -  2025-05-05
executed -  2025-05-05


In [19]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/prev_segment'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [20]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_regularity_segment
0,2025-03-31,63621000192f9565bfe3c91a,DAILY


In [21]:
df_previous_segment_concat.shape

(60488342, 3)

### Stability

In [22]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [23]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY
1,2025-03-03,5fec62a5ee2e9cf63c81f3d0,WEEKLY
2,2025-03-03,6260f2fb2c04373d294067d4,RARE_NEED
3,2025-03-03,63d8ae8f6b21d714418dfa55,WEEKLY
4,2025-03-03,659e4952b3bae65867b994db,RARE_NEED
...,...,...,...
5729986,2025-03-03,66ece87d7a4c8a6141bac9a9,RARE_NEED
5729987,2025-03-03,61b2daeebd965e1f5fbcefa3,RARE_NEED
5729988,2025-03-03,6460a8f39bb65e1f9021ee64,MONTHLY
5729989,2025-03-03,61585820c94e7965abcd3146,RARE_NEED


In [24]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-10,6720a5f8b0cf4681ad13ec44,BI_WEEKLY
1,2025-03-10,61cf75d1a75746bfa8e44619,RARE_NEED
2,2025-03-10,620b355057d2b2ceeeafedf8,RARE_NEED
3,2025-03-10,64aecfb77fe9533b3ff6af57,MONTHLY
4,2025-03-10,61b867c417dd6a10c413e7fa,WEEKLY
...,...,...,...
54758346,2025-05-05,66abf5a7c9ff56fcce918df4,WEEKLY
54758347,2025-05-05,62c2718c5bce425623ae2591,RARE_NEED
54758348,2025-05-05,5e6b3350f5667d7cb3239387,RARE_NEED
54758349,2025-05-05,64d5e6e412dbf310dfa2a262,RARE_NEED


In [25]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [26]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_regularity_segment_x,run_date_y,taxi_regularity_segment_y
0,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-10,MONTHLY
1,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-17,MONTHLY
2,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-24,MONTHLY
3,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-31,MONTHLY
4,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-04-07,MONTHLY
...,...,...,...,...,...
50084312,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-07,RARE_NEED
50084313,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-14,RARE_NEED
50084314,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-21,RARE_NEED
50084315,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-28,RARE_NEED


In [27]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_regularity_segment_x,taxi_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,756971
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,9220
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,115954
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,77517
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,5320
5,2025-03-03,2025-03-10,DAILY,DAILY,313881
6,2025-03-03,2025-03-10,DAILY,WEEKLY,14562
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,109612
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,660047
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,104682


In [28]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                     2025-03-10   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   756971.0   
                                     DAILY                         9220.0   
                                     MONTHLY                     115954.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       77517.0   
           DAILY                     BI_WEEKLY                     5320.0   
                                     DAILY                       313881.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       14562.0   
           MONTHLY                   BI_WEEKLY                   109612.0   
                                     DAILY                            0.0   
                                     MONTHLY                     660047.0   
                                     RARE_NEED                   104682.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                        0.0   
                                     DAILY                            0.0   
                                     MONTHLY                      82384.0   
                                     RARE_NEED                  2716145.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    68762.0   
                                     DAILY                        18447.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                      614960.0   

                                                                           \
run_date_y                                                     2025-03-17   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   633212.0   
                                     DAILY                        16382.0   
                                     MONTHLY                     185302.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                      113711.0   
           DAILY                     BI_WEEKLY                     9866.0   
                                     DAILY                       296106.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       23890.0   
           MONTHLY                   BI_WEEKLY                   153905.0   
                                     DAILY                            0.0   
                                     MONTHLY                     527646.0   
                                     RARE_NEED                   184614.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                    16567.0   
                                     DAILY                            0.0   
                                     MONTHLY                     130372.0   
                                     RARE_NEED                  2639717.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    96637.0   
                                     DAILY                 

In [91]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0).to_clipboard(index=False)

In [30]:
df_base_prev.groupby(['run_date', 'taxi_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,974871
1,2025-03-03,DAILY,340655
2,2025-03-03,MONTHLY,884696
3,2025-03-03,RARE_NEED,2812984
4,2025-03-03,WEEKLY,716785


### FE Regularity

In [31]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [34]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        -- taxi_regularity_segment
        taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = '{city_name}'
        and taxi_fe_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn3)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + city + '/prev_segment2/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [ ]:
for i in run_date:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-03
executed -  2025-03-03
starting -  2025-03-10
executed -  2025-03-10
starting -  2025-03-17
executed -  2025-03-17
starting -  2025-03-24


In [28]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/prev_segment2'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [29]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-31,6474a5c8694393cc7e442e77,DAILY


In [30]:
df_previous_segment_concat.shape

(35532535, 3)

### Stability

In [31]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [32]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY
1,2025-03-03,61dbe366fb8e4d610931ab58,BI_WEEKLY
2,2025-03-03,661a4d851f79f98055656dc3,MONTHLY
3,2025-03-03,5e33d682c725ca6c303e47e2,RARE_NEED
4,2025-03-03,61619a34aa0ecd1f9713558f,BI_WEEKLY
...,...,...,...
3448101,2025-03-03,65d0a2f26e764329106ed317,MONTHLY
3448102,2025-03-03,66384d8d5b3860598dc602e1,BI_WEEKLY
3448103,2025-03-03,5d844ff1af794421566a1ef1,WEEKLY
3448104,2025-03-03,6585c70a1093bb34fc3a09a0,WEEKLY


In [33]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-10,5abdecea25abd634611d51aa,MONTHLY
1,2025-03-10,62a735f41420e29d95401dc1,MONTHLY
2,2025-03-10,67aa9d57fe551003ad9fe5d9,MONTHLY
3,2025-03-10,67c341027c3ccf7a49876869,RARE_NEED
4,2025-03-10,6730dedc8ea7d581ee78aa92,BI_WEEKLY
...,...,...,...
32084424,2025-05-05,67e38cabd25c76821790bfd1,RARE_NEED
32084425,2025-05-05,6583d607cfe5392da9d31334,WEEKLY
32084426,2025-05-05,639b49dbd79523ee6cce4771,WEEKLY
32084427,2025-05-05,647dd4970b7c4a5f730081c2,RARE_NEED


In [34]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [35]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_fe_regularity_segment_x,run_date_y,taxi_fe_regularity_segment_y
0,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-10,BI_WEEKLY
1,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-17,BI_WEEKLY
2,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-24,BI_WEEKLY
3,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-31,MONTHLY
4,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-04-07,MONTHLY
...,...,...,...,...,...
26076936,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-07,BI_WEEKLY
26076937,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-14,MONTHLY
26076938,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-21,MONTHLY
26076939,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-28,MONTHLY


In [36]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_fe_regularity_segment_x,taxi_fe_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,852070
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,9595
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,119637
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,55537
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,6707
5,2025-03-03,2025-03-10,DAILY,DAILY,263169
6,2025-03-03,2025-03-10,DAILY,WEEKLY,14437
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,119766
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,782052
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,101823


In [37]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                           2025-03-10   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                      852070.0   
                                        DAILY                            9595.0   
                                        MONTHLY                        119637.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          55537.0   
           DAILY                        BI_WEEKLY                        6707.0   
                                        DAILY                          263169.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          14437.0   
           MONTHLY                      BI_WEEKLY                      119766.0   
                                        DAILY                               0.0   
                                        MONTHLY                        782052.0   
                                        RARE_NEED                      101823.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                           0.0   
                                        DAILY                               0.0   
                                        MONTHLY                         68392.0   
                                        RARE_NEED                      425258.0   
                                        WEEKLY                              0.0   
           WEEKLY                       BI_WEEKLY                       55659.0   
                                        DAILY                           13805.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                         380275.0   

                                                                                 \
run_date_y                                                           2025-03-17   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                      744171.0   
                                        DAILY                           17325.0   
                                        MONTHLY                        179325.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          84522.0   
           DAILY                        BI_WEEKLY                       11878.0   
                                        DAILY                          247903.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          22008.0   
           MONTHLY                      BI_WEEKLY                      180067.0   
                                        DAILY                               0.0   
                                        MONTHLY                        621075.0   
                                        RARE_NEED                      153928.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                         486.0   
                                        DAILY                               0.0   
                                        MONTHLY                         98306.0   
  

In [38]:
df_base_prev.groupby(['run_date', 'taxi_fe_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_fe_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,1056482
1,2025-03-03,DAILY,292637
2,2025-03-03,MONTHLY,1047096
3,2025-03-03,RARE_NEED,590726
4,2025-03-03,WEEKLY,461165
